# Lesson 01 - מבוא לסוכני בינה מלאכותית

ברוכים הבאים לשיעור הראשון בקורס **סוכני בינה מלאכותית למתחילים**!

**סוכן בינה מלאכותית** הוא תוכנה המשתמשת במודל שפה גדול (LLM) כמנוע ההסקה שלה ויכולה לבצע *פעולות* בעולם האמיתי — קריאת APIים, שאילת בסיסי נתונים, או הרצת קוד — כדי להשיג מטרה מטעם משתמש.

בעמוד הערות זה תבנו את הסוכן הראשון שלכם: **סוכן נסיעות** שממליץ על יעדי חופשה. במהלך הדרך תלמדו כיצד:

1. להתחבר לשירות סוכנים Azure AI Foundry באמצעות **מסגרת הסוכן של מיקרוסופט**.
2. לתת לסוכן **כלי** — פונקציית פייתון פשוטה שהוא יכול לקרוא לה.
3. להריץ את הסוכן ולבדוק את התגובה שלו.
4. להזרים את תגובת הסוכן טוקן אחר טוקן.


## התקנה

לפני הרצת מחברת זו, וודא שיש ברשותך:

1. **פרויקט Azure AI Foundry** עם מודל שיחה מופעל (למשל `gpt-4o-mini`).
2. **כניסה עם Azure CLI** — הרץ `az login` במסוף שלך.
3. **הגדרת משתני סביבה דרושים:**
   - `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Azure AI Foundry שלך.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהפעלת.

התא הבא מתקין את חבילות הפייתון שאתה צריך.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## יצירת הסוכן הראשון שלך

סוכן צריך שני דברים:

- **הוראות** שאומרות לו *מי הוא* ו*איך להתנהג* (פשוט מערכת).
- **כלים** — פונקציות פייתון המעוטרות ב-`@tool` שהסוכן יכול לקרוא כדי לקבל מידע או לבצע פעולות.

להלן אנו מגדירים כלי פשוט שמחזיר רשימה של יעדי חופשה פופולריים. הסוכן ישתמש בכלי הזה כאשר משתמש יבקש המלצות טיולים.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## תגובות בזרם

לחוויה אינטראקטיבית יותר, ניתן **לשדר** את תגובת הסוכן. במקום להמתין לתשובה המלאה, הסוכן מפיק קטעי טקסט בזמן שהם נוצרים. זה שימושי במיוחד בממשקי צ'אט שבהם רוצים להציג תוצאות בזמן אמת.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## סיכום

בשיעור זה למדת כיצד:

- **ליצור ספק** שמתחבר לשירות Azure AI Foundry Agent באמצעות `AzureAIProjectAgentProvider`.
- **להגדיר כלי** באמצעות הדקורטור `@tool` כך שהסוכן יוכל לקרוא לפונקציות הפייתון שלך.
- **להפעיל את הסוכן** עם הודעת משתמש ולהדפיס את תגובתו.
- **לשדר תגובות** לפלט בזמן אמת.

בשיעור הבא נחקור מסגרות סוכן בצורה מעמיקה יותר ונלמד כיצד לתת לסוכנים כלים רבי עוצמה ויכולת הסקת מסקנות רב-שלבית.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לשים לב כי תרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להתייחס למסמך המקורי בשפת המקור כמקור המוסמך. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אנושי. אנו לא נושאים באחריות על הבנות שגויות או פירושים מוטעים הנובעים משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
